# Try Apache Beam - Python

In this notebook, we set up your development environment and work through a simple example using the [DirectRunner](https://beam.apache.org/documentation/runners/direct/). You can explore other runners with the [Beam Capatibility Matrix](https://beam.apache.org/documentation/runners/capability-matrix/).

To navigate through different sections, use the table of contents. From **View**  drop-down list, select **Table of contents**.

To run a code cell, you can click the **Run cell** button at the top left of the cell, or by select it and press **`Shift+Enter`**. Try modifying a code cell and re-running it to see what happens.

To learn more about Colab, see [Welcome to Colaboratory!](https://colab.sandbox.google.com/notebooks/welcome.ipynb).

### Download Data

In [6]:
import subprocess

def run(cmd):
    print(f'>> {cmd}')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

# Download the Iris dataset
run('mkdir -p data')
run('curl -L -o data/iris.csv https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv')
run('head -n 10 data/iris.csv')

>> mkdir -p data

>> curl -L -o data/iris.csv https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  3858  100  3858    0     0  33224      0 --:--:-- --:--:-- --:--:-- 33258

>> head -n 10 data/iris.csv
sepal_length,sepal_width,petal_length,petal_width,species
5.1,3.5,1.4,0.2,setosa
4.9,3.0,1.4,0.2,setosa
4.7,3.2,1.3,0.2,setosa
4.6,3.1,1.5,0.2,setosa
5.0,3.6,1.4,0.2,setosa
5.4,3.9,1.7,0.4,setosa
4.6,3.4,1.4,0.3,setosa
5.0,3.4,1.5,0.2,setosa
4.4,2.9,1.4,0.2,setosa



### Setup Pipeline

1. Reads in CSV file line by line
2. Parses each line into fields
3. Keys each record by speecies
4. Uses CombineFn to compute per species statistics
5. Formats and writes the output

In [7]:
import apache_beam as beam
import csv
import io

In [13]:
def parse_csv_line(line):
    reader = csv.reader(io.StringIO(line))
    fields = next(reader)
    try:
        sepal_length = float(fields[0])
        sepal_width = float(fields[1])
        petal_length = float(fields[2])
        petal_width = float(fields[3])
        species = fields[4]
        return (species, (sepal_length, sepal_width, petal_length, petal_width))
    except (ValueError, IndexError):
        return None
    
class AggregateSpeciesStatistics(beam.CombineFn):
    def create_accumulator(self):
        # (sum_sl, sum_sw, sum_pl, sum_pw, count, min_pl, max_pl)
        return (0.0, 0.0, 0.0, 0.0, 0, float('inf'), float('-inf'))
    
    def add_input(self, accumulator, element):
        sum_sl, sum_sw, sum_pl, sum_pw, count, min_pl, max_pl = accumulator
        sl, sw, pl, pw = element
        return (
            sum_sl + sl,
            sum_sw + sw,
            sum_pl + pl,
            sum_pw + pw,
            count + 1,
            min(min_pl, pl),
            max(max_pl, pl)
        )
    
    def merge_accumulators(self, accumulators):
        sums = zip(*accumulators)
        sum_sl, sum_sw, sum_pl, sum_pw, count, min_pl, max_pl = [list(s) for s in sums]
        return (
            sum(sum_sl),
            sum(sum_sw),
            sum(sum_pw),
            sum(sum_pl),
            sum(count),
            min(min_pl),
            max(max_pl)
        )
    
    def extract_output(self, accumulator):
        sum_sl, sum_sw, sum_pl, sum_pw, count, min_pl, max_pl = accumulator
        if count == 0:
            return {}
        return {
            'avg_sepal_length': round(sum_sl / count, 2),
            'avg_sepal_width': round(sum_sw / count, 2),
            'avg_petal_length': round(sum_pl / count, 2),
            'avg_petal_width': round(sum_pl / count, 2),
            'min_petal_length': min_pl,
            'max_petal_length': max_pl,
            'count': count
        }
    
def format_results(kv):
    species, stats = kv
    return (
        f"Species: {species}\n"
        f"Avg Sepal Length = {stats['avg_sepal_length']} cm\n"
        f"Avg Sepal Width = {stats['avg_sepal_width']} cm\n"
        f"Avg Petal Length = {stats['avg_petal_length']} cm\n"
        f"Avg Petal Width = {stats['avg_petal_width']} cm\n"
        f"Petal Length Range = {stats['min_petal_length']} - {stats['max_petal_length']} cm\n"
        f"Samples = {stats['count']}"
    )

In [14]:
with beam.Pipeline() as pipeline:
    results = (
        pipeline

        # Read lines from CSV
        | 'Read CSV' >> beam.io.ReadFromText('data/iris.csv', skip_header_lines=1)

        # Parse CSV lines into species and measurements
        | 'Parse CSV' >> beam.Map(parse_csv_line)

        # Filter out failed rows
        | 'Filter rows' >> beam.Filter(lambda x: x is not None)

        # Aggregate stats by species
        | 'Aggregate stats' >> beam.CombinePerKey(AggregateSpeciesStatistics())

        # Format results
        | 'Format results' >> beam.Map(format_results)

        # Output results
        | 'Outpus results' >> beam.io.WriteToText("outputs/iris_stats")
    )

In [15]:
# Display results
run(f'cat outputs/iris_stats-00000-of-*')

>> cat outputs/iris_stats-00000-of-*
Species: setosa
Avg Sepal Length = 5.01 cm
Avg Sepal Width = 3.43 cm
Avg Petal Length = 0.25 cm
Avg Petal Width = 0.25 cm
Petal Length Range = 1.0 - 1.9 cm
Samples = 50
Species: versicolor
Avg Sepal Length = 5.94 cm
Avg Sepal Width = 2.77 cm
Avg Petal Length = 1.33 cm
Avg Petal Width = 1.33 cm
Petal Length Range = 3.0 - 5.1 cm
Samples = 50
Species: virginica
Avg Sepal Length = 6.59 cm
Avg Sepal Width = 2.97 cm
Avg Petal Length = 2.03 cm
Avg Petal Width = 2.03 cm
Petal Length Range = 4.5 - 6.9 cm
Samples = 50

